# Kinthread: pretrain our own reader on English (free Colab)

Our v10 reader, from scratch, learns English before it learns the fact language (Furrer et al. 2020: pre-training beats specialized architectures on compositional semantic parsing; TinyStories: small models learn fluent English from simple, varied text). Text: TinyStories V2 (CDLA-Sharing-1.0) and FineWeb-Edu (ODC-By 1.0), prepared exactly like exam questions.

1. **Data** (once per Google account; a CPU runtime is enough and uses no GPU time): run the first three cells.
2. **Pretrain** (T4 GPU): Runtime -> Change runtime type -> T4 GPU, then Run all. It resumes from Drive after a disconnect.
3. Then `train_reader_colab.ipynb` fine-tunes with `INIT` = this run's `final.pt`.

In [ ]:
import os, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (data cells only)')
REPO = '/content/Kinthread-llm-model'
if not os.path.exists(REPO):
    !git clone --depth 1 -b main https://github.com/ArAnirudh2901/Kinthread-llm-model.git {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
assert os.path.ismount('/content/drive'), 'Google Drive did not mount: click Allow and run this cell again'
BASE = next(b for b in ('/content/drive/MyDrive/kinthread_shared', '/content/drive/MyDrive/kinthread') if os.path.isdir(b) or b.endswith('/kinthread'))   # the folder shared from aravallianirudh2005, when this account has its shortcut
DATA, RUN = f'{BASE}/pretrain', 'pre-v2'
ANNEAL = next(k for k in (f'{BASE}/pretrain_kin_v7', f'{BASE}/pretrain_kin_v4', f'{BASE}/pretrain_kin') if os.path.exists(f'{k}/manifest.json'))   # kinship text: v7 once data prep built it   # pre-v2: NorMuon + QK-norm, kinship text from dataset_v7
OUT = f'{BASE}/runs/{RUN}'
os.makedirs(OUT, exist_ok=True)
print('shards ->', DATA, '| checkpoints ->', OUT)

In [ ]:
# ~400M tokens, half stories, half web, plus 60M tokens of verified kinship questions for the final phase.
# Resumable (finished shards are kept). About 30-60 minutes on 2 CPUs; a CPU runtime is enough.
if not os.path.exists(f'{DATA}/manifest.json'):
    !python pretrain_data.py build --vocab data/subword_vocab.json --out {DATA} --tokens 400000000 --mix tiny=0.5,web=0.5 --web_shards 1
if not os.path.exists(f'{ANNEAL}/manifest.json'):
    !python pretrain_data.py build --vocab data/subword_vocab.json --out {ANNEAL} --tokens 60000000 --mix domain=1 --domain_src data/dataset_v7
!cat {DATA}/manifest.json {ANNEAL}/manifest.json

## Pretrain
~39M parameters, deep and thin (16 layers, 384 wide; MobileLLM), fp32 weights with fp16 compute, blocks of 512 tokens, 400M tokens. Prints throughput every 100 steps and held-out loss every 1,000; saves every 500 steps. The last 20% (the decay) mixes in kinship text half of each batch (annealing, as in SmolLM2, MiniCPM and OLMo 2).

In [ ]:
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'
!python pretrain_reader.py --data {DATA} --vocab data/subword_vocab.json --anneal {ANNEAL} --out {OUT} --tokens 4e8 --amp --resume --optim muon --qk_norm